In [ ]:
# ============================================================
# Cell 1
# Imports, Environment Setup & Global Configuration
# ============================================================

# ------------------------------
# Jupyter / plotting setup
# ------------------------------
%matplotlib inline

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ------------------------------
# Standard libraries
# ------------------------------
import os
import gc
import time
import math
import json
import random
import pickle
from pathlib import Path
from collections import Counter, defaultdict
from typing import List, Tuple

# ------------------------------
# Scientific stack
# ------------------------------
import numpy as np
import pandas as pd

# ------------------------------
# Visualization
# ------------------------------
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 100

# ------------------------------
# PyTorch
# ------------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

# ------------------------------
# Scikit-learn utilities
# ------------------------------
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.utils.class_weight import compute_class_weight

# ------------------------------
# Optional libraries
# ------------------------------
try:
    from umap import UMAP
    UMAP_AVAILABLE = True
except Exception:
    UMAP_AVAILABLE = False

# ------------------------------
# Device configuration
# ------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True
    try:
        torch.backends.cuda.matmul.allow_tf32 = True
    except Exception:
        pass

# ------------------------------
# Reproducibility
# ------------------------------
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ------------------------------
# Directory layout (CLEAN BASELINE)
# ------------------------------
RAW_DIR = Path("WESAD")
DATA_DIR = Path("data_preprocessed")
RESULTS_EDA = Path("results_eda")
DIAG_DIR = Path("diagnostics")
MODELS_DIR = Path("models_hybrid")
RESULTS_TRAIN = Path("results_training_hybrid")
RESULTS_EVAL = Path("results_evaluation")
PUB_DIR = Path("publication_artifacts")

# Ensure directories exist
for p in (
    DATA_DIR,
    RESULTS_EDA,
    DIAG_DIR,
    MODELS_DIR,
    RESULTS_TRAIN,
    RESULTS_EVAL,
    PUB_DIR,
):
    p.mkdir(parents=True, exist_ok=True)

# ------------------------------
# Global labels
# ------------------------------
LABEL_MAP = {1: 0, 2: 1, 3: 2, 4: 3}
CLASS_NAMES = {
    0: "baseline",
    1: "stress",
    2: "amusement",
    3: "meditation",
}

NUM_CLASSES = len(CLASS_NAMES)

# ------------------------------
# Small helpers
# ------------------------------
def save_json(path: Path, obj: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


def load_json(path: Path):
    with open(path, "r") as f:
        return json.load(f)


print("\n✅ Cell 1 complete: environment initialized and directories ready.")

In [ ]:
# ============================================================
# Cell 2
# STEP 1 — DATA PREPROCESSING
# Load WESAD .pkl files, align chest & wrist signals,
# downsample to 32 Hz, window into 10s segments,
# and save as *_combined.npz
# ============================================================

# ------------------------------
# Configuration
# ------------------------------
TARGET_RATE = 32          # Hz
WINDOW_SEC = 10           # seconds
OVERLAP = 0.5             # 50% overlap
MAJORITY_THRESHOLD = 0.6  # label dominance threshold

# ------------------------------
# Helper functions
# ------------------------------
def block_downsample_1d(arr, factor):
    if factor <= 1:
        return arr.copy()
    n = len(arr) // factor
    if n == 0:
        return np.array([], dtype=arr.dtype)
    return arr[: n * factor].reshape(n, factor).mean(axis=1)


def acc_magnitude(acc):
    return np.sqrt((acc ** 2).sum(axis=1))


def majority_label(labels, threshold=MAJORITY_THRESHOLD):
    labels = np.asarray(labels)
    counts = np.bincount(labels)
    maj = np.argmax(counts)
    if counts[maj] / len(labels) >= threshold:
        return maj
    return 0  # baseline fallback


# ------------------------------
# Subject-level processing
# ------------------------------
def process_subject(pkl_path: Path, out_dir=DATA_DIR):
    subject = pkl_path.stem
    print(f"\n▶ Processing subject {subject}")

    with open(pkl_path, "rb") as f:
        data = pickle.load(f, encoding="latin1")

    signals = data["signal"]
    labels = np.array(data["label"], dtype=np.int32)

    # ---- Chest signals (~700 Hz) ----
    chest = signals["chest"]
    ch_acc = np.array(chest["ACC"])
    ch_ecg = np.array(chest["ECG"]).squeeze()
    ch_eda = np.array(chest["EDA"]).squeeze()
    ch_resp = np.array(chest["Resp"]).squeeze()
    ch_temp = np.array(chest["Temp"]).squeeze()

    # ---- Wrist signals (~32 Hz) ----
    wrist = signals["wrist"]
    wr_acc = np.array(wrist["ACC"])
    wr_eda = np.array(wrist["EDA"]).squeeze()
    wr_temp = np.array(wrist["TEMP"]).squeeze()

    # Reference length (chest ECG)
    ref_len = len(ch_ecg)
    labels = labels[:ref_len]

    # Trim chest signals
    ch_acc = ch_acc[:ref_len]
    ch_ecg = ch_ecg[:ref_len]
    ch_eda = ch_eda[:ref_len]
    ch_resp = ch_resp[:ref_len]
    ch_temp = ch_temp[:ref_len]

    # ---- Upsample wrist signals to chest length ----
    def upsample(arr, target_len):
        if len(arr) < 2:
            return np.zeros(target_len)
        x_old = np.linspace(0, 1, len(arr))
        x_new = np.linspace(0, 1, target_len)
        return np.interp(x_new, x_old, arr)

    wr_acc = np.vstack([
        upsample(wr_acc[:, i], ref_len) for i in range(3)
    ]).T
    wr_eda = upsample(wr_eda, ref_len)
    wr_temp = upsample(wr_temp, ref_len)

    # ---- Derived channels ----
    ch_acc_mag = acc_magnitude(ch_acc)
    wr_acc_mag = acc_magnitude(wr_acc)

    # ---- Downsample chest to 32 Hz ----
    ds_factor = max(1, int(700 / TARGET_RATE))

    ch_ecg = block_downsample_1d(ch_ecg, ds_factor)
    ch_resp = block_downsample_1d(ch_resp, ds_factor)
    ch_eda = block_downsample_1d(ch_eda, ds_factor)
    ch_temp = block_downsample_1d(ch_temp, ds_factor)
    ch_acc_mag = block_downsample_1d(ch_acc_mag, ds_factor)

    wr_eda = block_downsample_1d(wr_eda, ds_factor)
    wr_temp = block_downsample_1d(wr_temp, ds_factor)
    wr_acc_mag = block_downsample_1d(wr_acc_mag, ds_factor)

    labels = block_downsample_1d(labels.astype(np.float32), ds_factor).astype(np.int32)

    # ---- Align all channels ----
    min_len = min(
        len(ch_ecg), len(ch_resp), len(ch_eda), len(ch_temp),
        len(ch_acc_mag), len(wr_eda), len(wr_temp),
        len(wr_acc_mag), len(labels)
    )

    X_all = np.stack(
        [
            wr_acc_mag[:min_len],
            wr_eda[:min_len],
            wr_temp[:min_len],
            ch_ecg[:min_len],
            ch_resp[:min_len],
            ch_acc_mag[:min_len],
            ch_eda[:min_len],
            ch_temp[:min_len],
        ],
        axis=1,
    )

    labels = labels[:min_len]

    # ---- Sliding windows ----
    win_len = TARGET_RATE * WINDOW_SEC
    step = int(win_len * (1 - OVERLAP))

    X_list, y_list = [], []

    for start in range(0, len(X_all) - win_len, step):
        end = start + win_len
        X_win = X_all[start:end]
        y_win = labels[start:end]
        y_win = np.array([LABEL_MAP.get(l, 0) for l in y_win])
        y_label = majority_label(y_win)

        X_list.append(X_win)
        y_list.append(y_label)

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int64)

    print(f"  ✓ {subject}: {X.shape}, label distribution: {Counter(y)}")

    out_path = out_dir / f"{subject}_combined.npz"
    np.savez_compressed(out_path, X=X, y=y)
    return out_path


# ------------------------------
# Run preprocessing
# ------------------------------
if not RAW_DIR.exists():
    raise FileNotFoundError("❌ WESAD directory not found.")

pkl_files = sorted(RAW_DIR.glob("S*.pkl"))
if not pkl_files:
    raise FileNotFoundError("❌ No .pkl files found in WESAD/")

for pkl in pkl_files:
    process_subject(pkl)

print("\n✅ Cell 2 complete: preprocessing finished.")
print("   Preprocessed files saved to:", DATA_DIR.resolve())


In [ ]:
# ============================================================
# Cell 3
# STEP 2 — EXPLORATORY DATA ANALYSIS (EDA)
# Inline plots + optional saving
# ============================================================

print("=" * 90)
print("STEP 2: EXPLORATORY DATA ANALYSIS")
print("=" * 90)

SAVE_PLOTS = True  # toggle saving on/off

# ------------------------------
# Helper: show inline + optionally save
# ------------------------------
def show_and_save(fig=None, path=None, dpi=300):
    if fig is None:
        fig = plt.gcf()
    plt.tight_layout()
    plt.show()
    if SAVE_PLOTS and path is not None:
        fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)

# ------------------------------
# Load preprocessed data summary
# ------------------------------
files = sorted(DATA_DIR.glob("*_combined.npz"))
if not files:
    raise FileNotFoundError("❌ No preprocessed files found.")

records = []
for f in files:
    arr = np.load(f)
    y = arr["y"]
    total = len(y)
    for cls, cnt in Counter(y.tolist()).items():
        records.append(
            {
                "subject": f.stem.replace("_combined", ""),
                "class": CLASS_NAMES[cls],
                "count": cnt,
                "pct": 100 * cnt / total,
            }
        )

summary_df = pd.DataFrame(records)
summary_df.to_csv(RESULTS_EDA / "class_distribution_per_subject.csv", index=False)

print(f"✓ Loaded {len(files)} subjects")

# ============================================================
# 1️⃣ Overall class distribution
# ============================================================
plt.figure(figsize=(6, 4))
sns.barplot(
    data=summary_df,
    x="class",
    y="count",
    estimator=sum,
    ci=None,
    palette="muted",
)
plt.title("Overall Class Distribution")
show_and_save(path=RESULTS_EDA / "class_distribution_overall.png")

# ============================================================
# 2️⃣ Per-subject class percentage heatmap
# ============================================================
pivot = summary_df.pivot_table(
    values="pct", index="subject", columns="class", fill_value=0
)

plt.figure(figsize=(8, 6))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="Blues")
plt.title("Per-Subject Class Distribution (%)")
show_and_save(path=RESULTS_EDA / "class_distribution_subject_heatmap.png")

# ============================================================
# 3️⃣ Channel statistics (sample subject)
# ============================================================
sample_file = files[0]
with np.load(sample_file) as arr:
    X, y = arr["X"], arr["y"]

n_channels = X.shape[-1]
CHANNEL_NAMES = [
    "wr_acc_mag", "wr_eda", "wr_temp",
    "ch_ecg", "ch_resp", "ch_acc_mag", "ch_eda", "ch_temp"
]

stats = []
for cls in np.unique(y):
    Xi = X[y == cls]
    for i, ch in enumerate(CHANNEL_NAMES):
        stats.append(
            {
                "class": CLASS_NAMES[cls],
                "channel": ch,
                "mean": Xi[..., i].mean(),
                "std": Xi[..., i].std(),
            }
        )

stats_df = pd.DataFrame(stats)
stats_df.to_csv(RESULTS_EDA / "channel_stats_sample_subject.csv", index=False)

plt.figure(figsize=(10, 5))
sns.barplot(
    data=stats_df,
    x="channel",
    y="mean",
    hue="class",
    errorbar=None,
    palette="Set2",
)
plt.xticks(rotation=45, ha="right")
plt.title(f"Channel Means — {sample_file.stem}")
show_and_save(path=RESULTS_EDA / "channel_means_sample_subject.png")

# ============================================================
# 4️⃣ Low-dimensional embedding (PCA + UMAP / t-SNE)
# ============================================================
MAX_SAMPLES = 1500
n = min(MAX_SAMPLES, len(y))
idx = np.random.choice(len(y), n, replace=False)

Xsub = X[idx].reshape(n, -1)
y_sub = y[idx]

Xsub = StandardScaler().fit_transform(Xsub)

n_pca = min(20, Xsub.shape[1])
Xp = PCA(n_components=n_pca, random_state=SEED).fit_transform(Xsub)

if UMAP_AVAILABLE:
    reducer = UMAP(n_components=2, random_state=SEED)
    emb_name = "UMAP"
else:
    reducer = TSNE(n_components=2, random_state=SEED, perplexity=30)
    emb_name = "t-SNE"

Xemb = reducer.fit_transform(Xp)

plt.figure(figsize=(6, 5))
sns.scatterplot(
    x=Xemb[:, 0],
    y=Xemb[:, 1],
    hue=[CLASS_NAMES[int(i)] for i in y_sub],
    s=12,
    alpha=0.8,
)
plt.title(f"{emb_name} Projection (Sample Subject)")
plt.legend(bbox_to_anchor=(1.05, 1))
show_and_save(path=RESULTS_EDA / "embedding_projection.png")

# ============================================================
# 5️⃣ Raw signal snapshot (~10 seconds)
# ============================================================
win_len = min(320, X.shape[1])

plt.figure(figsize=(12, 6))
for i, ch in enumerate(CHANNEL_NAMES):
    plt.plot(X[:win_len, i] + i * 10, label=ch)

plt.title("Raw Signal Snapshot (~10s window)")
plt.xlabel("Time Steps")
plt.legend(loc="upper right", fontsize=9)
show_and_save(path=RESULTS_EDA / "raw_signal_snapshot.png")

print("\n✅ Cell 3 complete: EDA finished.")
print("   Results saved to:", RESULTS_EDA.resolve())
print("=" * 90)


In [ ]:
# ============================================================
# Cell 4
# STEP 3 — MODEL ARCHITECTURE
# CNN → BiGRU → Multi-Head Attention → Classifier
# ============================================================

# ------------------------------
# Helper utilities
# ------------------------------
def adjust_num_heads(embed_dim: int, requested_heads: int) -> int:
    """
    Ensure embed_dim is divisible by num_heads.
    """
    if requested_heads <= 0:
        return 1
    if embed_dim % requested_heads == 0:
        return requested_heads
    for h in range(requested_heads, 0, -1):
        if embed_dim % h == 0:
            return h
    return 1


def init_weights(module):
    """
    Kaiming/Xavier initialization for Conv, Linear, GRU.
    """
    if isinstance(module, (nn.Conv1d, nn.Linear)):
        nn.init.kaiming_uniform_(module.weight, a=math.sqrt(5))
        if module.bias is not None:
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(module.weight)
            bound = 1 / math.sqrt(max(1, fan_in))
            nn.init.uniform_(module.bias, -bound, bound)

    elif isinstance(module, nn.GRU):
        for name, param in module.named_parameters():
            if "weight" in name:
                nn.init.xavier_uniform_(param)
            elif "bias" in name:
                nn.init.constant_(param, 0.0)


# ------------------------------
# CNN Front-End
# ------------------------------
class CNNFrontEnd(nn.Module):
    """
    1D CNN front-end.
    Input:  (B, T, C)
    Output: (B, T, F)
    """

    def __init__(self, in_channels, out_channels=64, kernel_size=3, dropout=0.2):
        super().__init__()
        pad = kernel_size // 2

        self.net = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size, padding=pad),
            nn.ReLU(),
            nn.Conv1d(out_channels, out_channels, kernel_size, padding=pad),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        # (B, T, C) → (B, C, T)
        x = x.permute(0, 2, 1)
        x = self.net(x)
        # (B, C, T) → (B, T, F)
        return x.permute(0, 2, 1)


# ------------------------------
# CNN → BiGRU → Attention Model
# ------------------------------
class CNNBiGRU_Attn(nn.Module):
    """
    Hybrid temporal model:
      CNN → BiGRU → Multi-head self-attention → Mean pooling → Classifier
    """

    def __init__(
        self,
        input_channels: int,
        cnn_ch: int = 64,
        gru_hidden: int = 128,
        gru_layers: int = 2,
        attn_heads: int = 4,
        dropout: float = 0.3,
        num_classes: int = NUM_CLASSES,
    ):
        super().__init__()

        # CNN
        self.cnn = CNNFrontEnd(
            input_channels, out_channels=cnn_ch, dropout=dropout
        )

        # BiGRU
        self.gru = nn.GRU(
            input_size=cnn_ch,
            hidden_size=gru_hidden,
            num_layers=gru_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if gru_layers > 1 else 0.0,
        )

        self.norm = nn.LayerNorm(gru_hidden * 2)

        # Attention
        embed_dim = gru_hidden * 2
        attn_heads = adjust_num_heads(embed_dim, attn_heads)

        self.attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=attn_heads,
            batch_first=True,
            dropout=0.1,
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

        self.apply(init_weights)

    def forward(self, x):
        """
        x: (B, T, C)
        """
        x = self.cnn(x)
        x, _ = self.gru(x)
        x = self.norm(x)

        # Self-attention
        attn_out, _ = self.attn(x, x, x, need_weights=False)

        # Temporal pooling
        pooled = attn_out.mean(dim=1)

        return self.classifier(pooled)


# ------------------------------
# Model builder
# ------------------------------
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def build_model(
    input_channels,
    cnn_ch=64,
    gru_hidden=128,
    gru_layers=2,
    attn_heads=4,
    dropout=0.3,
):
    model = CNNBiGRU_Attn(
        input_channels=input_channels,
        cnn_ch=cnn_ch,
        gru_hidden=gru_hidden,
        gru_layers=gru_layers,
        attn_heads=attn_heads,
        dropout=dropout,
    )
    print(f"✓ Model built — {count_parameters(model):,} trainable parameters")
    return model


print("\n✅ Cell 4 complete: model architecture defined.")

In [ ]:
# ============================================================
# Cell 5
# STEP 4 — LOSO TRAINING WITH BALANCED BATCH SAMPLING
# ============================================================

NUM_CLASSES = 4
BATCH_SIZE = 128
EPOCHS = 25
LR = 1e-3
PATIENCE = 6

# 🔴 Windows-safe settings
NUM_WORKERS = 0
PIN_MEMORY = False
GRAD_CLIP = 5.0


# ------------------------------------------------------------
# Streaming mean/std computation
# ------------------------------------------------------------
def compute_mean_std_streaming(npz_paths):
    total_count = 0
    sum_, sumsq = None, None

    for p in npz_paths:
        with np.load(p) as arr:
            X = arr["X"].astype(np.float64)

        flat = X.reshape(-1, X.shape[-1])
        cnt = flat.shape[0]

        s = flat.sum(axis=0)
        ss = (flat ** 2).sum(axis=0)

        sum_ = s if sum_ is None else sum_ + s
        sumsq = ss if sumsq is None else sumsq + ss
        total_count += cnt

    mean = (sum_ / total_count).astype(np.float32)
    var = (sumsq / total_count) - mean.astype(np.float64) ** 2
    std = np.sqrt(np.maximum(var, 1e-12)).astype(np.float32)

    return mean, std


# ------------------------------------------------------------
# Vectorized preload
# ------------------------------------------------------------
def preload_npz_files(npz_paths, mean, std):
    X_all, y_all = [], []

    for p in npz_paths:
        with np.load(p) as arr:
            X = arr["X"].astype(np.float32)
            y = arr["y"].astype(np.int64)

        X = (X - mean[None, None, :]) / (std[None, None, :] + 1e-9)

        X_all.append(torch.from_numpy(X))
        y_all.append(torch.from_numpy(y))

    return torch.cat(X_all), torch.cat(y_all)


class InMemoryDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# ------------------------------------------------------------
# 🔴 Balanced sampler (INTEGRATED HERE)
# ------------------------------------------------------------
from torch.utils.data import WeightedRandomSampler

def make_balanced_sampler(y_tensor):
    y_np = y_tensor.numpy()
    class_counts = np.bincount(y_np, minlength=NUM_CLASSES)
    class_weights = 1.0 / np.maximum(class_counts, 1)

    sample_weights = class_weights[y_np]
    sample_weights = torch.tensor(sample_weights, dtype=torch.double)

    return WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
    )


# ------------------------------------------------------------
# Training loop
# ------------------------------------------------------------
def train_fold(train_ds, val_ds, model, fold_name):
    use_amp = DEVICE.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    sampler = make_balanced_sampler(train_ds.y)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        sampler=sampler,      # 🔴 BALANCED SAMPLING
        num_workers=0,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
    )

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

    best_val = float("inf")
    epochs_no_improve = 0

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            with torch.cuda.amp.autocast(enabled=use_amp):
                loss = criterion(model(xb), yb)

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * xb.size(0)

        train_loss /= len(train_ds)

        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE)
                with torch.cuda.amp.autocast(enabled=use_amp):
                    val_loss += criterion(model(xb), yb).item() * xb.size(0)

        val_loss /= len(val_ds)

        print(
            f"[{fold_name}] Epoch {epoch+1:02d} | "
            f"train={train_loss:.4f} | val={val_loss:.4f}"
        )

        if val_loss < best_val:
            best_val = val_loss
            epochs_no_improve = 0
            torch.save(
                {"model_state": model.state_dict()},
                MODELS_DIR / f"best_{fold_name}.pt",
            )
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"[{fold_name}] Early stopping")
                break


# ------------------------------------------------------------
# LOSO driver
# ------------------------------------------------------------
files = sorted(DATA_DIR.glob("*_combined.npz"))
assert files, "❌ No preprocessed files found"

for test_file in files:
    test_subj = test_file.stem.split("_")[0]
    print("\n" + "=" * 80)
    print("TEST SUBJECT:", test_subj)
    print("=" * 80)

    train_files = [f for f in files if f != test_file]

    mean, std = compute_mean_std_streaming(train_files)
    X_all, y_all = preload_npz_files(train_files, mean, std)

    idx = torch.randperm(len(X_all))
    n_val = int(0.1 * len(idx))
    val_idx, train_idx = idx[:n_val], idx[n_val:]

    train_ds = InMemoryDataset(X_all[train_idx], y_all[train_idx])
    val_ds = InMemoryDataset(X_all[val_idx], y_all[val_idx])

    model = build_model(input_channels=X_all.shape[2]).to(DEVICE)
    train_fold(train_ds, val_ds, model, test_subj)

    del model, X_all, y_all
    torch.cuda.empty_cache()
    gc.collect()

print("\n✅ LOSO TRAINING COMPLETE (BALANCED SAMPLING)")


In [ ]:
# ============================================================
# Cell 5b
# STEP 4B — BALANCED LOSO TRAINING + TEST EVALUATION
# (In-memory, Windows-safe, paper-ready)
# ============================================================

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

NUM_CLASSES = 4
BATCH_SIZE = 128
EPOCHS = 25
LR = 1e-3
PATIENCE = 6
GRAD_CLIP = 5.0
NUM_WORKERS = 0   # Windows-safe


# ------------------------------------------------------------
# Utilities
# ------------------------------------------------------------
def compute_mean_std_streaming(npz_paths):
    total = 0
    s, ss = None, None

    for p in npz_paths:
        with np.load(p) as arr:
            X = arr["X"].astype(np.float64)

        flat = X.reshape(-1, X.shape[-1])
        s = flat.sum(axis=0) if s is None else s + flat.sum(axis=0)
        ss = (flat ** 2).sum(axis=0) if ss is None else ss + (flat ** 2).sum(axis=0)
        total += flat.shape[0]

    mean = (s / total).astype(np.float32)
    var = (ss / total) - mean.astype(np.float64) ** 2
    std = np.sqrt(np.maximum(var, 1e-12)).astype(np.float32)
    return mean, std


def preload_npz_files(npz_paths, mean, std):
    Xs, ys = [], []
    for p in npz_paths:
        with np.load(p) as arr:
            X = arr["X"].astype(np.float32)
            y = arr["y"].astype(np.int64)

        X = (X - mean[None, None, :]) / (std[None, None, :] + 1e-9)
        Xs.append(torch.from_numpy(X))
        ys.append(torch.from_numpy(y))

    return torch.cat(Xs), torch.cat(ys)


class InMemoryDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def make_balanced_sampler(y):
    y_np = y.numpy()
    counts = np.bincount(y_np, minlength=NUM_CLASSES)
    weights = 1.0 / np.maximum(counts, 1)
    sample_weights = torch.tensor(weights[y_np], dtype=torch.double)
    return WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)


# ------------------------------------------------------------
# Training + evaluation per fold
# ------------------------------------------------------------
def train_and_evaluate_fold(
    train_ds, val_ds, test_ds, model, test_subj
):
    use_amp = DEVICE.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    sampler = make_balanced_sampler(train_ds.y)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    )
    test_loader = DataLoader(
        test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    )

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

    best_val = float("inf")
    epochs_no_improve = 0

    # ---------------- TRAIN ----------------
    for epoch in range(EPOCHS):
        model.train()
        train_loss, n_train = 0.0, 0

        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            with torch.cuda.amp.autocast(enabled=use_amp):
                loss = criterion(model(xb), yb)

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item() * xb.size(0)
            n_train += xb.size(0)

        train_loss /= n_train

        # -------- VALIDATION --------
        model.eval()
        val_loss, n_val = 0.0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                with torch.cuda.amp.autocast(enabled=use_amp):
                    loss = criterion(model(xb), yb)
                val_loss += loss.item() * xb.size(0)
                n_val += xb.size(0)

        val_loss /= n_val

        print(
            f"[{test_subj}] Epoch {epoch+1:02d} | "
            f"train={train_loss:.4f} | val={val_loss:.4f}"
        )

        if val_loss < best_val:
            best_val = val_loss
            epochs_no_improve = 0
            torch.save(
                {"model_state": model.state_dict()},
                MODELS_DIR / f"best_{test_subj}.pt",
            )
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f"[{test_subj}] Early stopping")
                break

    # ---------------- TEST ----------------
    ckpt = torch.load(MODELS_DIR / f"best_{test_subj}.pt", map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    preds, targets = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            preds.append(torch.argmax(logits, dim=1).cpu())
            targets.append(yb)

    y_true = torch.cat(targets).numpy()
    y_pred = torch.cat(preds).numpy()

    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average="macro", zero_division=0)
    cm = confusion_matrix(y_true, y_pred)

    report = classification_report(
        y_true,
        y_pred,
        target_names=[CLASS_NAMES[i] for i in range(NUM_CLASSES)],
        output_dict=True,
        zero_division=0,
    )

    save_json(
        RESULTS_EVAL / f"eval_{test_subj}.json",
        {
            "subject": test_subj,
            "accuracy": float(acc),
            "f1_macro": float(f1m),
            "confusion_matrix": cm.tolist(),
            "classification_report": report,
        },
    )

    print(f"[{test_subj}] TEST | acc={acc:.4f} | f1_macro={f1m:.4f}")


# ------------------------------------------------------------
# LOSO driver
# ------------------------------------------------------------
files = sorted(DATA_DIR.glob("*_combined.npz"))
assert files, "❌ No preprocessed files found."

for test_file in files:
    test_subj = test_file.stem.split("_")[0]
    print("\n" + "=" * 80)
    print(f"TEST SUBJECT: {test_subj}")
    print("=" * 80)

    train_files = [f for f in files if f != test_file]

    mean, std = compute_mean_std_streaming(train_files)

    X_train_all, y_train_all = preload_npz_files(train_files, mean, std)
    with np.load(test_file) as arr:
        X_test = torch.from_numpy(
            (arr["X"] - mean[None, None, :]) / (std[None, None, :] + 1e-9)
        )
        y_test = torch.from_numpy(arr["y"])

    idx = torch.randperm(len(X_train_all))
    n_val = int(0.1 * len(idx))
    val_idx, train_idx = idx[:n_val], idx[n_val:]

    train_ds = InMemoryDataset(X_train_all[train_idx], y_train_all[train_idx])
    val_ds = InMemoryDataset(X_train_all[val_idx], y_train_all[val_idx])
    test_ds = InMemoryDataset(X_test, y_test)

    model = build_model(input_channels=X_train_all.shape[2]).to(DEVICE)

    train_and_evaluate_fold(
        train_ds, val_ds, test_ds, model, test_subj
    )

    del model, X_train_all, y_train_all, X_test, y_test
    torch.cuda.empty_cache()
    gc.collect()

print("\n✅ Balanced LOSO training + evaluation COMPLETE")

In [ ]:
# ============================================================
# Cell 6a
# STEP 5 — BASELINE EVALUATION & VISUALIZATION
# (Uses results from Cell 5)
# ============================================================

print("=" * 90)
print("STEP 5: BASELINE EVALUATION (NO BALANCING)")
print("=" * 90)

SAVE_PLOTS = True

# ------------------------------
# Helper: inline + optional save
# ------------------------------
def show_and_save(fig=None, path=None):
    if fig is None:
        fig = plt.gcf()
    plt.tight_layout()
    plt.show()
    if SAVE_PLOTS and path is not None:
        fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)


# ------------------------------
# Load per-subject results
# ------------------------------
result_files = sorted(RESULTS_TRAIN.glob("results_*.json"))
assert result_files, "❌ No training results found. Run Cell 5 first."

fold_results = []
for rf in result_files:
    with open(rf) as f:
        fold_results.append(json.load(f))

print(f"✓ Loaded results for {len(fold_results)} subjects")

# ------------------------------
# Aggregate per-subject metrics
# ------------------------------
summary_rows = []
for res in fold_results:
    summary_rows.append(
        {
            "subject": res["subject"],
            "accuracy": res["acc"],
            "f1_macro": res["f1_macro"],
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(RESULTS_EVAL / "baseline_summary_per_subject.csv", index=False)

mean_acc = summary_df["accuracy"].mean()
std_acc = summary_df["accuracy"].std()
mean_f1 = summary_df["f1_macro"].mean()
std_f1 = summary_df["f1_macro"].std()

print(f"\nBaseline LOSO Results:")
print(f"  Mean Accuracy : {mean_acc:.4f} ± {std_acc:.4f}")
print(f"  Mean Macro-F1 : {mean_f1:.4f} ± {std_f1:.4f}")

# ============================================================
# 1️⃣ Accuracy & Macro-F1 per subject
# ============================================================
plt.figure(figsize=(8, 4))
sns.barplot(data=summary_df, x="subject", y="accuracy", color="skyblue")
plt.ylim(0, 1)
plt.title("Baseline Accuracy per Subject")
show_and_save(path=RESULTS_EVAL / "baseline_accuracy_per_subject.png")

plt.figure(figsize=(8, 4))
sns.barplot(data=summary_df, x="subject", y="f1_macro", color="salmon")
plt.ylim(0, 1)
plt.title("Baseline Macro-F1 per Subject")
show_and_save(path=RESULTS_EVAL / "baseline_f1_per_subject.png")

# ============================================================
# 2️⃣ Aggregated confusion matrix
# ============================================================
cm_total = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)

for res in fold_results:
    cm_total += np.array(res["confusion_matrix"])

cm_norm = cm_total / cm_total.sum(axis=1, keepdims=True)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=CLASS_NAMES.values(),
    yticklabels=CLASS_NAMES.values(),
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Baseline Confusion Matrix (Normalized)")
show_and_save(path=RESULTS_EVAL / "baseline_confusion_matrix.png")

# ============================================================
# 3️⃣ Per-class F1 (PROMINENT)
# ============================================================
class_f1 = {name: [] for name in CLASS_NAMES.values()}

for res in fold_results:
    report = res["report"]
    for cname in class_f1:
        if cname in report:
            class_f1[cname].append(report[cname]["f1-score"])

class_f1_df = pd.DataFrame(
    {
        "class": list(class_f1.keys()),
        "mean_f1": [np.mean(v) for v in class_f1.values()],
        "std_f1": [np.std(v) for v in class_f1.values()],
    }
)

class_f1_df.to_csv(
    RESULTS_EVAL / "baseline_per_class_f1.csv", index=False
)

plt.figure(figsize=(6, 4))
sns.barplot(
    data=class_f1_df,
    x="class",
    y="mean_f1",
    palette="viridis",
)
plt.ylim(0, 1)
plt.ylabel("F1 score")
plt.title("Baseline Per-Class F1 (LOSO)")
show_and_save(path=RESULTS_EVAL / "baseline_per_class_f1.png")

# ------------------------------
# Text summary (useful for paper)
# ------------------------------
print("\nPer-class F1 (Baseline):")
for _, r in class_f1_df.iterrows():
    print(f"  {r['class']:<12s}: {r['mean_f1']:.3f}")

print("\n✅ Cell 6a complete: baseline evaluation & visualization done.")
print("=" * 90)

In [ ]:
# ============================================================
# Cell 6b
# STEP 5 — BALANCED EVALUATION & VISUALIZATION
# (Uses RESULTS_EVAL only; saves classification_report)
# ============================================================

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

print("=" * 90)
print("STEP 5: BALANCED EVALUATION & VISUALIZATION")
print("=" * 90)

SAVE_PLOTS = True


# ------------------------------------------------------------
# Helper: inline + optional save
# ------------------------------------------------------------
def show_and_save(fig=None, path=None):
    if fig is None:
        fig = plt.gcf()
    plt.tight_layout()
    plt.show()
    if SAVE_PLOTS and path is not None:
        fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)


# ------------------------------------------------------------
# Load balanced evaluation results
# ------------------------------------------------------------
balanced_files = sorted(RESULTS_EVAL.glob("eval_*.json"))
assert balanced_files, "❌ No balanced evaluation results found. Run Cell 5b first."

balanced_results = []
for f in balanced_files:
    with open(f) as fh:
        balanced_results.append(json.load(fh))

print(f"✓ Loaded balanced results for {len(balanced_results)} subjects")


# ------------------------------------------------------------
# Aggregate per-subject metrics
# ------------------------------------------------------------
rows = []
for res in balanced_results:
    rows.append(
        {
            "subject": res["subject"],
            "accuracy": res["accuracy"],
            "f1_macro": res["f1_macro"],
        }
    )

summary_df = pd.DataFrame(rows)
summary_df.to_csv(
    RESULTS_EVAL / "balanced_summary_per_subject.csv", index=False
)

mean_acc = summary_df["accuracy"].mean()
std_acc = summary_df["accuracy"].std()
mean_f1 = summary_df["f1_macro"].mean()
std_f1 = summary_df["f1_macro"].std()

print("\nBalanced LOSO Results:")
print(f"  Mean Accuracy : {mean_acc:.4f} ± {std_acc:.4f}")
print(f"  Mean Macro-F1 : {mean_f1:.4f} ± {std_f1:.4f}")


# ============================================================
# 1️⃣ Accuracy & Macro-F1 per subject (Balanced)
# ============================================================
plt.figure(figsize=(8, 4))
sns.barplot(data=summary_df, x="subject", y="accuracy", color="skyblue")
plt.ylim(0, 1)
plt.title("Balanced Accuracy per Subject")
show_and_save(path=RESULTS_EVAL / "balanced_accuracy_per_subject.png")

plt.figure(figsize=(8, 4))
sns.barplot(data=summary_df, x="subject", y="f1_macro", color="salmon")
plt.ylim(0, 1)
plt.title("Balanced Macro-F1 per Subject")
show_and_save(path=RESULTS_EVAL / "balanced_f1_per_subject.png")


# ============================================================
# 2️⃣ Aggregated confusion matrix (Balanced)
# ============================================================
cm_total = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)

for res in balanced_results:
    cm_total += np.array(res["confusion_matrix"])

cm_norm = cm_total / cm_total.sum(axis=1, keepdims=True)

labels = [CLASS_NAMES[i] for i in range(NUM_CLASSES)]

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels,
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Balanced Confusion Matrix (Normalized)")
show_and_save(path=RESULTS_EVAL / "balanced_confusion_matrix.png")


# ============================================================
# 3️⃣ Per-class F1 (PROMINENT, Balanced)
# ============================================================
class_f1 = {name: [] for name in CLASS_NAMES.values()}

for res in balanced_results:
    if "classification_report" not in res:
        continue   # skip old files safely

    report = res["classification_report"]
    for cname in class_f1:
        if cname in report:
            class_f1[cname].append(report[cname]["f1-score"])


class_f1_df = pd.DataFrame(
    {
        "class": list(class_f1.keys()),
        "mean_f1": [np.mean(v) for v in class_f1.values()],
        "std_f1": [np.std(v) for v in class_f1.values()],
    }
)

class_f1_df.to_csv(
    RESULTS_EVAL / "balanced_per_class_f1.csv", index=False
)

plt.figure(figsize=(6, 4))
sns.barplot(
    data=class_f1_df,
    x="class",
    y="mean_f1",
    palette="viridis",
)
plt.ylim(0, 1)
plt.ylabel("F1 score")
plt.title("Balanced Per-Class F1 (LOSO)")
show_and_save(path=RESULTS_EVAL / "balanced_per_class_f1.png")


# ------------------------------------------------------------
# Text summary (paper-ready)
# ------------------------------------------------------------
print("\nPer-class F1 (Balanced):")
for _, r in class_f1_df.iterrows():
    print(f"  {r['class']:<12s}: {r['mean_f1']:.3f}")

print("\n✅ Cell 6b complete: balanced evaluation & visualization done.")
print("=" * 90)

In [ ]:
# ============================================================
# Cell 6c
# STEP 6 — BASELINE vs BALANCED COMPARISON (FIXED & FINAL)
# ============================================================

print("=" * 90)
print("STEP 6: BASELINE vs BALANCED PERFORMANCE COMPARISON")
print("=" * 90)

SAVE_PLOTS = True

# ------------------------------------------------------------
# Helper: inline + optional save
# ------------------------------------------------------------
def show_and_save(fig=None, path=None):
    if fig is None:
        fig = plt.gcf()
    plt.tight_layout()
    plt.show()
    if SAVE_PLOTS and path is not None:
        fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)


# ------------------------------------------------------------
# Load BASELINE results (Cell 5)
# ------------------------------------------------------------
baseline_files = sorted(RESULTS_TRAIN.glob("results_*.json"))
assert baseline_files, "❌ Baseline results not found. Run Cell 5."

baseline_results = []
for path in baseline_files:
    with open(path, "r") as fh:
        baseline_results.append(json.load(fh))

print(f"✓ Loaded baseline results for {len(baseline_results)} subjects")


# ------------------------------------------------------------
# Load BALANCED results (Cell 5b)
# ------------------------------------------------------------
balanced_files = sorted(RESULTS_EVAL.glob("eval_*.json"))
assert balanced_files, "❌ Balanced results not found. Run Cell 5b."

balanced_results = []
for path in balanced_files:
    with open(path, "r") as fh:
        balanced_results.append(json.load(fh))

print(f"✓ Loaded balanced results for {len(balanced_results)} subjects")


# ------------------------------------------------------------
# Per-subject Accuracy & Macro-F1 comparison
# ------------------------------------------------------------
rows = []
for b, bal in zip(baseline_results, balanced_results):
    assert b["subject"] == bal["subject"], "Subject mismatch"

    rows.append(
        {
            "subject": b["subject"],
            "Baseline Accuracy": b["acc"],
            "Balanced Accuracy": bal["accuracy"],
            "Baseline F1": b["f1_macro"],
            "Balanced F1": bal["f1_macro"],
        }
    )

cmp_df = pd.DataFrame(rows)
cmp_df.to_csv(RESULTS_EVAL / "baseline_vs_balanced_summary.csv", index=False)

# ---------------- Accuracy plot ----------------
acc_melt = cmp_df.melt(
    id_vars="subject",
    value_vars=["Baseline Accuracy", "Balanced Accuracy"],
    var_name="setting",
    value_name="accuracy",
)

plt.figure(figsize=(9, 4))
sns.barplot(data=acc_melt, x="subject", y="accuracy", hue="setting")
plt.ylim(0, 1)
plt.title("Baseline vs Balanced Accuracy (LOSO)")
show_and_save(path=RESULTS_EVAL / "compare_accuracy.png")

# ---------------- Macro-F1 plot ----------------
f1_melt = cmp_df.melt(
    id_vars="subject",
    value_vars=["Baseline F1", "Balanced F1"],
    var_name="setting",
    value_name="f1_macro",
)

plt.figure(figsize=(9, 4))
sns.barplot(data=f1_melt, x="subject", y="f1_macro", hue="setting")
plt.ylim(0, 1)
plt.title("Baseline vs Balanced Macro-F1 (LOSO)")
show_and_save(path=RESULTS_EVAL / "compare_f1_macro.png")


# ------------------------------------------------------------
# Per-class F1 comparison (CRITICAL FOR PAPER)
# ------------------------------------------------------------
def extract_class_f1(results):
    out = {c: [] for c in CLASS_NAMES.values()}
    for r in results:
        rep = r["report"] if "report" in r else r["classification_report"]
        for cname in out:
            if cname in rep:
                out[cname].append(rep[cname]["f1-score"])
    return {k: np.mean(v) for k, v in out.items()}

baseline_class_f1 = extract_class_f1(baseline_results)
balanced_class_f1 = extract_class_f1(balanced_results)

class_cmp_df = pd.DataFrame(
    {
        "class": list(CLASS_NAMES.values()),
        "Baseline": [baseline_class_f1[c] for c in CLASS_NAMES.values()],
        "Balanced": [balanced_class_f1[c] for c in CLASS_NAMES.values()],
    }
)

class_cmp_df.to_csv(
    RESULTS_EVAL / "baseline_vs_balanced_per_class_f1.csv", index=False
)

class_melt = class_cmp_df.melt(
    id_vars="class",
    value_vars=["Baseline", "Balanced"],
    var_name="setting",
    value_name="f1",
)

plt.figure(figsize=(6, 4))
sns.barplot(data=class_melt, x="class", y="f1", hue="setting")
plt.ylim(0, 1)
plt.ylabel("F1 score")
plt.title("Per-Class F1: Baseline vs Balanced (LOSO)")
show_and_save(path=RESULTS_EVAL / "compare_per_class_f1.png")


# ------------------------------------------------------------
# Text summary (for Results section)
# ------------------------------------------------------------
print("\nPer-class F1 comparison:")
for c in CLASS_NAMES.values():
    print(
        f"  {c:<12s} | "
        f"Baseline={baseline_class_f1[c]:.3f} | "
        f"Balanced={balanced_class_f1[c]:.3f}"
    )

print("\n✅ Cell 6c complete — results ready for paper.")
print("=" * 90)